# 04 · Synergistic Proximity

Computes distance from each establishment to the nearest bus stop, hospital, school, and park using Overpass API.

**Input:** `csv/00_base_data.csv`
**Output:** `csv/04_synergistic_proximity.csv` — `osm_id`, `dist_bus_stop_m`, `dist_hospital_m`, `dist_school_m`, `dist_park_m`

In [ ]:
import overpy, requests, pandas as pd, math, time

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

# Proximity categories
PROXIMITY_TAGS = {
    "bus_stop": [
        ("highway", "bus_stop"),
        ("amenity", "bus_station"),
        ("public_transport", "stop_position"),
    ],
    "hospital": [
        ("amenity", "hospital"),
        ("amenity", "clinic"),
    ],
    "school": [
        ("amenity", "school"),
        ("amenity", "university"),
        ("amenity", "college"),
    ],
    "park": [
        ("leisure", "park"),
        ("leisure", "garden"),
    ],
}

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def overpass_query(query):
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            return overpy.Overpass().parse_json(r.text)
        except Exception as e:
            print(f"  x {ep}: {e}")
            time.sleep(3)
    raise RuntimeError("All Overpass endpoints failed.")


def fetch_proximity_pois(lat, lon, radius):
    lines = []
    tag_to_cat = {}
    for cat, pairs in PROXIMITY_TAGS.items():
        for key, val in pairs:
            tag_to_cat[(key, val)] = cat
            lines.append(f'node["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')
            lines.append(f'way["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')
    query = "[out:json][timeout:90];\n(\n    " + "\n    ".join(lines) + "\n);\nout center tags;"

    print(f"Fetching proximity POIs (radius {radius:.0f} m)...")
    result = overpass_query(query)

    pois = {cat: [] for cat in PROXIMITY_TAGS}
    for el in list(result.nodes) + list(result.ways):
        if hasattr(el, "lat"):
            elat, elon = float(el.lat), float(el.lon)
        elif hasattr(el, "center_lat") and el.center_lat:
            elat, elon = float(el.center_lat), float(el.center_lon)
        else:
            continue
        for (key, val), cat in tag_to_cat.items():
            if el.tags.get(key) == val:
                pois[cat].append((elat, elon))
                break

    for cat, items in pois.items():
        print(f"  {cat:10s}: {len(items)} POIs")
    return pois


def compute_nearest_distances(df, pois):
    for cat, poi_list in pois.items():
        dists = []
        for _, row in df.iterrows():
            if not poi_list:
                dists.append(None)
                continue
            best = min(haversine(row["lat"], row["lon"], p[0], p[1]) for p in poi_list)
            dists.append(round(best, 1))
        df[f"dist_{cat}_m"] = dists
        filled = [d for d in dists if d is not None]
        if filled:
            print(f"  dist_{cat}_m: min {min(filled):.0f}  avg {sum(filled)/len(filled):.0f}  max {max(filled):.0f}")
    return df

print("Functions ready.")

In [ ]:
# Infer origin and search radius from base data
origin_lat = (df["lat"].max() + df["lat"].min()) / 2
origin_lon = (df["lon"].max() + df["lon"].min()) / 2

# Search radius: cover all shops + 1500 m buffer
max_dist = max(
    haversine(origin_lat, origin_lon, row["lat"], row["lon"])
    for _, row in df.iterrows()
)
search_radius = max_dist + 1500

pois = fetch_proximity_pois(origin_lat, origin_lon, search_radius)

print("\nComputing nearest distances...")
df = compute_nearest_distances(df, pois)

In [ ]:
df_out = df[["osm_id", "dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m"]]
df_out.to_csv("csv/04_synergistic_proximity.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/04_synergistic_proximity.csv")
print(df_out.describe().round(1))